# Práctica 1 - Pipeline alternativo de Machine Learning

En esta práctica se construye un pipeline alternativo completo de Machine Learning para predecir el impago de préstamos. Para ello, se implementa una nueva clase de preprocesamiento y una nueva clase de filtrado, siguiendo el patrón `fit/transform` para evitar data leakage. Posteriormente, se entrenan tres modelos de distintas familias: un ensemble de árboles, una máquina de soporte vectorial y una red neuronal. Finalmente, se comparan sus resultados con el modelo base visto en clase.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import precision_recall_curve, auc

from src.preprocessing.practica1_preprocessing import Practica1Preprocess
from src.filtering.practica1_filtering import Practica1Filtering

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (practica1_preprocessing.py, line 501)

## 1. Preprocesamiento de los datos

En primer lugar, se aplica la clase `Practica1Preprocess`, que utiliza el fichero `variables_withExperts.xlsx` para incluir también las variables procedentes de evaluaciones de expertos. Además, se emplean técnicas alternativas a las de la clase base para la imputación de valores nulos, el tratamiento de variables categóricas, el escalado de variables numéricas y la creación de nuevas features.

In [ ]:
preprocessor = Practica1Preprocess(
    var_to_process="data/variables_withExperts.xlsx",
    target="loan_status"
)

preprocessor.fit("data/df_train_small.csv")

X_train_pre, y_train = preprocessor.transform("data/df_train_small.csv")
X_test_pre, y_test = preprocessor.transform("data/df_test_small.csv")

preprocessor.print_summary()

print("Shape X_train después de preprocesamiento:", X_train_pre.shape)
print("Shape X_test después de preprocesamiento:", X_test_pre.shape)
print("Shape y_train:", y_train.shape)
print("Shape y_test:", y_test.shape)

RESUMEN DEL PREPROCESAMIENTO PRACTICA 1
Variables predictoras usadas: 102
Variables numéricas originales: 92
Variables categóricas originales: 15
Variables ordinales: ['grade', 'sub_grade']
Variables con frequency encoding: 13
Variables numéricas finales escaladas: 107
Shape X_train después de preprocesamiento: (80000, 107)
Shape X_test después de preprocesamiento: (20000, 107)
Shape y_train: (80000,)
Shape y_test: (20000,)


c:\Users\sofia\Desktop\CUARTO CURSO\Modelizacion en Ingenieria de datos\Modelizacion-de-datos2\.venv\Lib\site-packages\feature_engine\encoding\base_encoder.py:260: UserWarning: During the encoding, NaN values were introduced in the feature(s) emp_title, desc, zip_code, addr_state, earliest_cr_line, sec_app_earliest_cr_line.
  warnings.warn(


## 2. Filtrado de variables

Una vez preprocesados los datos, se aplica la clase `Practica1Filtering`. En este caso, se ha optado por un filtrado alternativo basado en dos pasos: eliminación de variables con varianza muy baja y selección de las variables más relevantes mediante información mutua. Este proceso permite reducir la dimensionalidad y conservar las características con mayor poder predictivo.

In [ ]:
print("NaN en X_train_pre:", X_train_pre.isna().sum().sum())
print("NaN en X_test_pre:", X_test_pre.isna().sum().sum())

NaN en X_train_pre: 0
NaN en X_test_pre: 0


In [ ]:
filtering = Practica1Filtering(variance_threshold=0.0, k_best=100)

filtering.fit(X_train_pre, y_train)

X_train_filt = filtering.transform(X_train_pre)
X_test_filt = filtering.transform(X_test_pre)

filtering.print_summary()

print("Shape X_train después de filtrado:", X_train_filt.shape)
print("Shape X_test después de filtrado:", X_test_filt.shape)

RESUMEN DEL FILTRADO PRACTICA 1
Features iniciales:                    107
Tras filtro de varianza:              107
Features seleccionadas finales:       100
Shape X_train después de filtrado: (80000, 100)
Shape X_test después de filtrado: (20000, 100)


## 3. Entrenamiento de modelos

Una vez finalizado el preprocesamiento y el filtrado, se entrenan tres modelos de aprendizaje automático pertenecientes a familias distintas. En concreto, se utilizará un ensemble de árboles de decisión, una máquina de soporte vectorial y una red neuronal. El objetivo es comparar su rendimiento en la predicción de impago sobre el conjunto de test.

### 3.1 Ensemble de árboles

Como modelo de tipo ensemble se ha elegido un `RandomForestClassifier`, ya que suele ofrecer un buen rendimiento en problemas tabulares, permite capturar relaciones no lineales entre variables y es relativamente robusto frente a ruido y outliers. Además, se ha configurado con `class_weight='balanced'` para tener en cuenta el desbalanceo entre clases.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_filt, y_train.values.ravel());

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",20
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

### 3.2 Máquina de soporte vectorial

Como representante de las máquinas de soporte vectorial se emplea un modelo `SVC` con kernel radial (`rbf`). Este tipo de modelo puede capturar fronteras de decisión no lineales y resulta útil como comparación frente a los modelos basados en árboles. Se activa la opción `probability=True` para poder calcular posteriormente la métrica PR-AUC.

In [ ]:
# Para que no sea excesivamente lento, se toma una muestra del train
X_train_svm = X_train_filt.sample(n=10000, random_state=42)
y_train_svm = y_train.loc[X_train_svm.index]

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    class_weight="balanced",
    random_state=42
)

svm_model.fit(X_train_svm, y_train_svm.values.ravel());

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


### 3.3 Red neuronal

Como tercer modelo se entrena una red neuronal multicapa mediante `MLPClassifier`. Este modelo permite aproximar relaciones complejas entre variables y constituye una tercera familia claramente distinta a las anteriores. Se ha escogido una arquitectura sencilla y un número moderado de iteraciones para mantener un equilibrio entre rendimiento y tiempo de entrenamiento.

In [ ]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    learning_rate_init=0.001,
    max_iter=400,
    early_stopping=True,
    random_state=42
)

mlp_model.fit(X_train_filt, y_train.values.ravel());

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",400
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42


## 4. Evaluación de modelos

Para comparar los modelos entrenados, se calculan las métricas solicitadas en el conjunto de test: `accuracy`, `precision`, `recall` y `PR-AUC`. Dado que el problema presenta clases desbalanceadas, la métrica PR-AUC resulta especialmente relevante, ya que se centra en la capacidad del modelo para detectar correctamente la clase minoritaria, es decir, los casos de impago.

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_true = y_test.values.ravel()
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)

    precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall_vals, precision_vals)

    results = {
        "Modelo": model_name,
        "Accuracy": accuracy,
        "Precision (impago)": precision,
        "Recall (impago)": recall,
        "PR-AUC": pr_auc
    }

    return results

In [ ]:
results_rf = evaluate_model(rf_model, X_test_filt, y_test, "Random Forest")
results_svm = evaluate_model(svm_model, X_test_filt, y_test, "SVM")
results_mlp = evaluate_model(mlp_model, X_test_filt, y_test, "MLP")

In [ ]:
base_results = {
    "Modelo": "Modelo base",
    "Accuracy": 0.72,
    "Precision (impago)": 0.26,
    "Recall (impago)": 0.24,
    "PR-AUC": np.nan
}

results_df = pd.DataFrame([
    base_results,
    results_rf,
    results_svm,
    results_mlp
])

results_df

,Modelo,Accuracy,Precision (impago),Recall (impago),PR-AUC
0,Modelo base,0.7200,0.260000,0.240000,NaN
1,Random Forest,0.6943,0.341802,0.572179,0.380653
2,SVM,0.6932,0.228897,0.225919,0.229500
3,MLP,0.7976,0.463415,0.080811,0.291866


In [ ]:
results_df.style.format({
    "Accuracy": "{:.3f}",
    "Precision (impago)": "{:.3f}",
    "Recall (impago)": "{:.3f}",
    "PR-AUC": "{:.3f}"
})

,Modelo,Accuracy,Precision (impago),Recall (impago),PR-AUC
0,Modelo base,0.720,0.260,0.240,nan
1,Random Forest,0.694,0.342,0.572,0.381
2,SVM,0.693,0.229,0.226,0.230
3,MLP,0.798,0.463,0.081,0.292


## 5. Comparación con el modelo base

La tabla de resultados muestra diferencias claras entre los modelos entrenados y el modelo base de referencia. Este baseline obtenía aproximadamente un 72% de accuracy, un 26% de precision en la clase de impago y un 24% de recall.

El modelo que mejor detecta los casos de impago es el **Random Forest**, ya que alcanza un recall de 0.580, muy superior al 0.240 del modelo base. Además, también mejora la precision, pasando de 0.260 a 0.342, y obtiene una PR-AUC de 0.379, lo que indica un mejor rendimiento global sobre la clase minoritaria. Su principal inconveniente es que la accuracy baja ligeramente hasta 0.693, lo que sugiere que detecta más impagos a costa de cometer más errores globales.

El modelo **SVM** ofrece un rendimiento inferior al baseline en casi todas las métricas relevantes. Su accuracy es de 0.693, su precision de 0.229 y su recall de 0.226, valores que no mejoran los del modelo base. Además, la PR-AUC de 0.228 confirma que su capacidad para discriminar los casos de impago es bastante limitada en este problema.

Por su parte, la **red neuronal MLP** obtiene la mejor accuracy de todos los modelos, con un valor de 0.790, y también mejora la precision respecto al baseline, alcanzando 0.375. Sin embargo, su recall cae hasta 0.078, lo que significa que apenas detecta una pequeña parte de los impagos reales. Por tanto, aunque acierta mucho en términos globales, no resulta especialmente útil si el objetivo principal es identificar clientes con riesgo de impago.

En conjunto, estos resultados muestran que el mejor modelo depende de la métrica que se considere prioritaria. Si el objetivo es maximizar la detección de impagos, el Random Forest es claramente la mejor opción. Si se prioriza la accuracy global, el MLP obtiene el mejor resultado, aunque a costa de perder gran parte de la capacidad de detección de la clase minoritaria.

## 6. Conclusiones

En esta práctica se ha construido un pipeline alternativo de Machine Learning para la predicción de impago, utilizando una estrategia de preprocesamiento, filtrado y modelado distinta a la empleada en clase. En particular, se han incorporado variables de expertos, se han aplicado métodos alternativos de imputación, codificación y escalado, y se han generado nuevas variables derivadas con significado financiero.

Tras el preprocesamiento y el filtrado, se obtuvo un conjunto final de 100 variables, con el que se entrenaron tres modelos de familias distintas: un Random Forest, una máquina de soporte vectorial y una red neuronal MLP. Esta comparación ha permitido comprobar que no existe un único modelo mejor en todas las métricas, sino que el rendimiento depende del criterio de evaluación considerado.

El modelo **Random Forest** ha sido el más equilibrado para este problema, ya que mejora claramente al modelo base en precision, recall y PR-AUC, especialmente en la detección de impagos. Aunque su accuracy es algo inferior a la del baseline, ofrece un comportamiento mucho más útil desde el punto de vista del riesgo, porque identifica una proporción bastante mayor de clientes problemáticos.

La **SVM** no ha conseguido mejorar el rendimiento del modelo base, lo que puede deberse a que este tipo de modelo es más sensible a la elección de hiperparámetros y al tamaño de la muestra utilizada en el entrenamiento. En cuanto al **MLP**, aunque ha alcanzado la mayor accuracy, su recall ha sido demasiado bajo, lo que indica que tiende a clasificar la mayoría de los casos como no impago y, por tanto, no resulta adecuado si se busca detectar de forma efectiva la clase minoritaria.

En conclusión, el pipeline propuesto sí permite mejorar el modelo base, pero dicha mejora depende del objetivo concreto. Si lo más importante es detectar impagos, el Random Forest es la alternativa más adecuada entre las probadas. Estos resultados también ponen de manifiesto la importancia de evaluar modelos desbalanceados con métricas más allá de la accuracy, ya que una accuracy alta no garantiza una buena detección de la clase de interés.